# Radio channels

> What the radio channels of both levels share: the fading, the power control, the interference and the airtime around their receiver.

In [ ]:
#| default_exp radio.channel

In [ ]:
#| hide
from nbdev.showdoc import *

A radio channel is a pipeline of blocks, one call of each per step, with one row per transmission:

| Block | Gives | |
|---|---|---|
| `Fading` | `h` | at each subcarrier |
| `PowerControl` | `power`, `active` | outages: not active |
| `Interference` | `interference`, `active`, `half_duplex` | only with `interference=True` |
| the receiver | `success`, ... | the subclass's: what arrives |
| `Airtime` | `latency`, `energy` | symbols / bandwidth, power × time |

`RadioChannel` is what both levels share: `h`, the power, the interference and the airtime. The receiver is where
they differ:

| channel | its receiver |
|---|---|
| `SystemLevelChannel` | `SINR` → `ErrorModel` → `Decide`: the PHY abstraction maps the SINR to a packet error rate |
| `LinkLevelChannel` | `BitLevelLink`: the bits, through `y = h √P x + n` with the interference added to the noise |
| `SionnaLinkChannel` | `SINR` → `SionnaLink`: the SINR is link adaptation's (the MCS), then the coded bits |

At the link level the receiver works on the signal itself, so it needs no SINR: the SNR it operates at follows
from `h`, `P` and the noise. The SINR is needed where something is decided from it, a packet error rate or an MCS.

A subclass gives its receiver (`receiver`). `pipeline()` returns the
blocks, so they can be reused: in a `PipelineBackend` (see `net.backend`), with a block replaced or added, e.g. a Sionna
block in `Named`. The channel builds its pipeline from its attributes (`fading`, `power_control`, ...) at every
call, so changing one changes the channel.

**The results** of both levels report the SNR, `P |h|² / noise_power`, computed from `h` and the power whatever the
receiver; the energy (power × duration) and the latency (symbols / bandwidth, 0 with an infinite bandwidth). Their
`metadata` holds `channel_gain_db` and `tx_power`, and `outage` for links the power control did not use
(`success=False`, 0 bits sent).

With interference (`interference=True`), a result's `snr_db` is the SINR, `P |h|² / (noise_power +
interference)`, and the metadata has `interference_power`.

Sionna has no such object: its tutorials write the loop from the channel to the PHY abstraction themselves. Here a
radio channel is that loop, for the transmissions of a `Network` step.

In [ ]:
#| export
from __future__ import annotations

import math
from typing import List, Sequence, Tuple

import numpy as np

from comm_core.net.backend import STEP_INPUTS, step_inputs, to_results
from comm_core.block import Block, Pipeline, _child_seed
from comm_core.net.channel import Channel
from comm_core.core import Transmission, TransmissionResult
from comm_core.radio.interference import Interference
from comm_core.radio.power_control import FixedPower, PowerControl
from comm_core.radio.fading import Fading, NoFading
from comm_core.net.topology import Link

In [ ]:
#| export
class Airtime(Block):
    "Time on the air (symbols / bandwidth; 0 with an infinite bandwidth) and transmit energy (power × time; power × symbols without a bandwidth)."

    outputs = ('latency', 'energy')

    def call(
        self,
        num_symbols: np.ndarray, # Symbols each transmission puts on the air
        power: np.ndarray, # Their transmit powers
        link: Sequence[Link], # Their links (the bandwidth)
    ) -> Tuple[np.ndarray, np.ndarray]:
        n = len(link)
        latency, energy = np.zeros(n), np.zeros(n)
        for k in range(n):
            symbols, bandwidth, p = int(num_symbols[k]), link[k].bandwidth_hz, float(power[k])
            duration = symbols / bandwidth if math.isfinite(bandwidth) else 0.0
            latency[k], energy[k] = duration, p * (duration if duration else symbols)
        return latency, energy


class RadioChannel(Channel):
    "A channel over radio links, made of blocks: a `Fading` (h), a `PowerControl` (P), a receiver (the subclass's), the airtime."

    def __init__(
        self,
        fading: Fading | None = None, # The coefficient h of a transmission; None: `NoFading()` (h = 1)
        power_control: PowerControl | None = None, # Transmit power and outages; None: `FixedPower()` (the link's tx_power_w)
        noise_power: float = 1.0, # Noise power at the receiver (same unit as the power)
        seed: int | None = None, # Seed of the channel's randomness and of the fading
        interference: bool = False, # A batch's transmissions share the medium: each one's SINR counts the others' power
        half_duplex: bool = True, # With interference: a node that transmits in a step cannot receive in it
    ):
        self.fading = fading if fading is not None else NoFading()
        self.power_control = power_control if power_control is not None else FixedPower()
        self.noise_power = noise_power
        self.interference, self.half_duplex = interference, half_duplex
        self.airtime = Airtime()
        self._pipelines, self._interference = {}, None
        self.reset(seed)

    def receiver(self) -> List[Block]:
        "The blocks between the power control (or the interference) and the airtime, which decide what arrives (the last one draws it)."
        raise NotImplementedError

    def blocks(self) -> List[Block]:
        "The channel's blocks, in order; with `interference`, an `Interference` (of the channel's fading) after the power control."
        between = []
        if self.interference:
            i = self._interference
            if i is None or i.fading is not self.fading or i.half_duplex != self.half_duplex:
                self._interference = i = Interference(self.fading, self.half_duplex)
            between = [i]
        return [self.fading, self.power_control, *between, *self.receiver(), self.airtime]

    def pipeline(self) -> Pipeline:
        "The channel as a `Pipeline` of its blocks (see `block`), with the step's inputs and `noise_power`."
        blocks = self.blocks()
        key = tuple(map(id, blocks))
        if key not in self._pipelines:
            self._pipelines = {key: Pipeline(blocks, inputs=STEP_INPUTS + ('noise_power',))}
        return self._pipelines[key]

    def transmit(
        self,
        transmission: Transmission,
        link: Link,
    ) -> TransmissionResult:
        return self.transmit_batch([(transmission, link)])[0]

    def transmit_batch(
        self,
        transmissions: List[Tuple[Transmission, Link]], # Transmissions, with their links
    ) -> List[TransmissionResult]:
        "Several transmissions, with one call of each block; with `interference`, at the same time (a `Network` step's)."
        if not transmissions:
            return []
        return to_results(self.pipeline()(**step_inputs(transmissions), noise_power=self.noise_power))

    def reset(self, seed: int | None = None) -> None:
        "Start over: `seed` reseeds the receiver, the fading and the power control, each with its own seed; the power control forgets its state."
        *others, draw = self.receiver()
        draw.reset(_child_seed(seed, 0))
        self.fading.reset(_child_seed(seed, 1))
        self.power_control.reset(_child_seed(seed, 2))
        for i, block in enumerate(others):
            block.reset(_child_seed(seed, 3 + i))

    def update(self, time: float) -> None:
        "A new round: passed to the blocks (e.g. `MapChannelInversion` notes where the nodes are)."
        for block in self.blocks():
            block.update(time)

    def set_positions(self, positions) -> None:
        "Where the nodes are, for a fading that depends on it (e.g. `RayTracedFading`)."
        if hasattr(self.fading, 'set_positions'):
            self.fading.set_positions(positions)

In [ ]:
show_doc(Airtime.call)

In [ ]:
show_doc(RadioChannel.pipeline)

In [ ]:
show_doc(RadioChannel.receiver)

In [ ]:
latency, energy = Airtime()(np.array([100, 100]), np.array([0.5, 0.5]), [Link('a', 'b', bandwidth_hz=1e6), Link('a', 'b')])
assert latency.tolist() == [1e-4, 0.0] and energy.tolist() == [0.5e-4, 50.0]    # no bandwidth: power x symbols

## The medium

With `interference=True`, the transmissions a channel gets at once (`transmit_batch`: a `Network` step's, through
`AnalyticalBackend`) share one medium: they happen at the same time and interfere. The channel runs its blocks with an
`Interference` block after the power control. See `interference` for what it models: the other senders' power at
each receiver, half-duplex radios (`half_duplex`), open-loop power control. A transmission alone (`transmit`) has
nothing to interfere with.

It works at both levels: `SystemLevelChannel` uses the SINR for the packet error rate, `LinkLevelChannel` adds the
interference to the noise of every bit, and `SionnaLinkChannel` to the noise of every resource element (and picks
the MCS from the SINR).

Whether to avoid simultaneous transmissions is the `Protocol`'s job: `TDMAProtocol` gives each node its own
slots, and `SlottedALOHA` transmits at random (see `net.protocol`). With `IdealProtocol`, everything sent before a step
goes out in that step, so if every node talks at once, nobody hears (half-duplex).

### Tests

In [ ]:
from comm_core.core import Message
from comm_core.radio.power_control import ChannelInversion
from comm_core.radio.ray_tracing import RayTracedFading
from comm_core.radio.system.channel import SystemLevelChannel

In [ ]:
# 21 positions on a line, |H|^2 = 1e-6 / d^3 (d in cells)
xs = np.arange(21)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
H = np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0)
table = RayTracedFading(H, [(x, 0) for x in xs])
ch = SystemLevelChannel(fading=table, noise_power=1e-10, per_model='threshold', snr_threshold_db=10, seed=0, interference=True)
def tx(s, r, bits=64):
    return Transmission(message=Message(s, r, 'hi', size_bits=bits), sender=s, receiver=r, start_time=0.0), Link(s, r)

table.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (20, 0), 'd': (19, 0)})
alone, = ch.transmit_batch([tx('a', 'b')])
assert alone.success and abs(alone.snr_db - 40) < 1e-9                 # 1 W x 1e-6 / 1e-10: 40 dB
# c -> d far away: a little interference at b
far = ch.transmit_batch([tx('a', 'b'), tx('c', 'd')])
i = 1e-6 / 19 ** 3
assert all(r.success for r in far) and abs(far[0].metadata['interference_power'] - i) < 1e-20
assert abs(far[0].snr_db - 10 * np.log10(1e-6 / (1e-10 + i))) < 1e-9
# c -> d next to b: a-b drops to 9 dB, below the threshold; c-d (a is 4 cells from d) still gets through
table.set_positions({'c': (3, 0), 'd': (4, 0)})
near = ch.transmit_batch([tx('a', 'b'), tx('c', 'd')])
assert not near[0].success and 8.9 < near[0].snr_db < 9.1 and near[1].success
assert ch.transmit(*tx('a', 'b')).success                                 # alone: nothing to interfere with
quiet = SystemLevelChannel(fading=table, noise_power=1e-10, per_model='threshold', snr_threshold_db=10)
assert all(r.success for r in quiet.transmit_batch([tx('a', 'b'), tx('c', 'd')]))   # without interference: each as if alone
# half-duplex: a and b talk to each other at once, neither hears
both = ch.transmit_batch([tx('a', 'b'), tx('b', 'a')])
assert not any(r.success for r in both) and all(r.metadata['half_duplex'] for r in both) and both[0].size_bits == 64
ch.half_duplex = False
assert all(r.success for r in ch.transmit_batch([tx('a', 'b'), tx('b', 'a')]))
ch.half_duplex = True
# one sender, two receivers: no interference with itself
two = ch.transmit_batch([tx('a', 'b'), tx('a', 'c')])
assert all(r.success for r in two) and 'interference_power' not in two[0].metadata
# outages are not on the air: they don't interfere
ch_inv = SystemLevelChannel(fading=table, power_control=ChannelInversion(20, max_power=1e-6), noise_power=1e-10, interference=True)
r = ch_inv.transmit_batch([tx('a', 'b'), tx('c', 'd')])
assert all(x.metadata.get('outage') for x in r)                        # each needs 1e-2 W > 1e-6 W
assert [type(b).__name__ for b in ch.pipeline()][:4] == ['RayTracedFading', 'FixedPower', 'Interference', 'SINR']
assert [type(b).__name__ for b in quiet.pipeline()][:3] == ['RayTracedFading', 'FixedPower', 'SINR']

In a `Network`, a step's transmissions are the ones that happen at the same time:

In [ ]:
from comm_core import Network, Node, FullMeshTopology, IdealProtocol, AnalyticalBackend

In [ ]:
nodes = ['a', 'b', 'c', 'd']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), IdealProtocol(), AnalyticalBackend(ch))
table.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (20, 0), 'd': (19, 0)})
net.send(Message('a', 'b', 'x', size_bits=64))
net.send(Message('c', 'd', 'y', size_bits=64))
assert [r.success for r in net.step()] == [True, True] and net.receive('b')[0].payload == 'x'
# everybody talks at once: every node is transmitting, so nobody hears
for s in nodes:
    for r in nodes:
        if s != r:
            net.send(Message(s, r, 'z', size_bits=64))
assert not any(r.success for r in net.step())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()